O serviço de vendas de carros usados Rusty Bargain está desenvolvendo um aplicativo para atrair novos clientes. Nesse aplicativo, você pode descobrir rapidamente o valor de mercado do seu carro. Você tem acesso a dados históricos: especificações técnicas, versões de acabamento e preços. Você precisa construir o modelo para determinar o valor. 

Rusty Bargain está interessado em:

- a qualidade da predição;
- a velocidade da predição;
- o tempo necessário para o treinamento

## Preparação de Dados

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

In [2]:
df = pd.read_csv('/datasets/car_data.csv')
print(df.shape)
df.head()

(354369, 16)


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

In [4]:
print('Valores ausentes:')
print(df.isnull().sum())

Valores ausentes:
DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64


In [5]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


Olhando os dados, vejo vários problemas que preciso tratar:

- Algumas colunas não ajudam a prever o preço: DateCrawled, DateCreated e LastSeen são datas de atividade do usuário, NumberOfPictures é sempre 0 e PostalCode é só localização. Vou remover todas.
- Há valores ausentes em VehicleType, Gearbox, Model, FuelType, Brand e NotRepaired (todas categóricas).
- RegistrationYear tem valores impossíveis (de 1000 a 9999). Carros não foram registrados no ano 1000 nem em 9999, então vou manter só de 1900 a 2016.
- Power tem muitos zeros e valores absurdos (até 20000 hp). Vou manter uma faixa realista de 10 a 1000 hp.
- Price tem zeros e valores muito baixos. Carro de 0 euro não faz sentido para treinar o modelo, então vou manter só preços de 100 euros para cima.

In [6]:
# remover colunas que não ajudam na previsão
df = df.drop(['DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode'], axis=1)

# filtrar valores impossíveis
df = df[df['Price'] >= 100]
df = df[(df['RegistrationYear'] >= 1900) & (df['RegistrationYear'] <= 2016)]
df = df[(df['Power'] >= 10) & (df['Power'] <= 1000)]

# preencher categóricas ausentes com 'unknown'
cat_cols = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
for c in cat_cols:
    df[c] = df[c].fillna('unknown')

# remover duplicados
df = df.drop_duplicates().reset_index(drop=True)

print('Formato após limpeza:', df.shape)
print('Nulos restantes:', df.isnull().sum().sum())

Formato após limpeza: (272162, 11)
Nulos restantes: 0


In [7]:
target = df['Price']
features = df.drop('Price', axis=1)

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=12345
)

print('Treino:', features_train.shape)
print('Teste:', features_test.shape)

Treino: (204121, 10)
Teste: (68041, 10)


Para os modelos mais simples (árvore de decisão, floresta aleatória e regressão linear) preciso converter as colunas categóricas em números. Vou usar OrdinalEncoder, ajustando só no treino para não vazar dados. O LightGBM lida com categóricas nativamente, então para ele vou usar uma versão separada com as colunas marcadas como category.

In [8]:
# versão com OrdinalEncoder para árvore, floresta e regressão linear
encoder = OrdinalEncoder()
features_train_enc = features_train.copy()
features_test_enc = features_test.copy()
features_train_enc[cat_cols] = encoder.fit_transform(features_train[cat_cols])
features_test_enc[cat_cols] = encoder.transform(features_test[cat_cols])

# versão com category nativo para o LightGBM
features_train_cat = features_train.copy()
features_test_cat = features_test.copy()
for c in cat_cols:
    features_train_cat[c] = features_train_cat[c].astype('category')
    features_test_cat[c] = features_test_cat[c].astype('category')

## Treinamento do modelo

Vou treinar os modelos do mais simples ao mais complexo, medindo em cada um o REQM, o tempo de treino e o tempo de predição.

In [9]:
# guardar resultados para comparar no final
resultados = []

def avaliar(nome, model, X_train, y_train, X_test, y_test):
    inicio = time.time()
    model.fit(X_train, y_train)
    tempo_treino = time.time() - inicio

    inicio = time.time()
    pred = model.predict(X_test)
    tempo_pred = time.time() - inicio

    rmse = mean_squared_error(y_test, pred) ** 0.5

    resultados.append({
        'Modelo': nome,
        'REQM': rmse,
        'Tempo treino (s)': tempo_treino,
        'Tempo predição (s)': tempo_pred
    })
    print(f'{nome}: REQM={rmse:.2f} | treino={tempo_treino:.2f}s | predição={tempo_pred:.4f}s')

### Regressão linear (prova real)

In [10]:
lr = LinearRegression()
avaliar('Regressão Linear', lr, features_train_enc, target_train, features_test_enc, target_test)

Regressão Linear: REQM=2953.27 | treino=0.04s | predição=0.0047s


### Árvore de decisão

In [11]:
for depth in [5, 10, 15]:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=12345)
    avaliar(f'Árvore de Decisão (depth={depth})', dt,
            features_train_enc, target_train, features_test_enc, target_test)

Árvore de Decisão (depth=5): REQM=2418.38 | treino=0.30s | predição=0.0054s
Árvore de Decisão (depth=10): REQM=1963.19 | treino=0.37s | predição=0.0079s
Árvore de Decisão (depth=15): REQM=1902.48 | treino=0.51s | predição=0.0122s


### Floresta aleatória

In [12]:
for n_est in [40, 80]:
    rf = RandomForestRegressor(n_estimators=n_est, max_depth=15,
                               random_state=12345, n_jobs=-1)
    avaliar(f'Floresta Aleatória (n={n_est})', rf,
            features_train_enc, target_train, features_test_enc, target_test)

Floresta Aleatória (n=40): REQM=1642.66 | treino=6.89s | predição=0.1966s
Floresta Aleatória (n=80): REQM=1636.20 | treino=13.52s | predição=0.3987s


### LightGBM

O LightGBM recebe as colunas categóricas diretamente, sem precisar de codificação manual. Testei alguns conjuntos de hiperparâmetros.

In [13]:
configs = [
    {'n_estimators': 100, 'learning_rate': 0.1, 'num_leaves': 31},
    {'n_estimators': 200, 'learning_rate': 0.1, 'num_leaves': 31},
    {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63},
]

for cfg in configs:
    lgbm = lgb.LGBMRegressor(random_state=12345, verbose=-1, **cfg)
    inicio = time.time()
    lgbm.fit(features_train_cat, target_train, categorical_feature=cat_cols)
    tempo_treino = time.time() - inicio
    inicio = time.time()
    pred = lgbm.predict(features_test_cat)
    tempo_pred = time.time() - inicio
    rmse = mean_squared_error(target_test, pred) ** 0.5
    nome = f"LightGBM (n={cfg['n_estimators']}, lr={cfg['learning_rate']}, leaves={cfg['num_leaves']})"
    resultados.append({'Modelo': nome, 'REQM': rmse,
                       'Tempo treino (s)': tempo_treino, 'Tempo predição (s)': tempo_pred})
    print(f'{nome}: REQM={rmse:.2f} | treino={tempo_treino:.2f}s | predição={tempo_pred:.4f}s')

/.venv/lib/python3.9/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')


LightGBM (n=100, lr=0.1, leaves=31): REQM=1601.69 | treino=1.82s | predição=0.3069s


/.venv/lib/python3.9/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')


LightGBM (n=200, lr=0.1, leaves=31): REQM=1571.51 | treino=3.10s | predição=0.5056s


/.venv/lib/python3.9/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')


LightGBM (n=500, lr=0.05, leaves=63): REQM=1536.34 | treino=9.31s | predição=1.6836s


## Análise do modelo

In [14]:
tabela = pd.DataFrame(resultados).sort_values('REQM').reset_index(drop=True)
tabela

,Modelo,REQM,Tempo treino (s),Tempo predição (s)
0,"LightGBM (n=500, lr=0.05, leaves=63)",1536.341450,9.312624,1.683615
1,"LightGBM (n=200, lr=0.1, leaves=31)",1571.508138,3.102505,0.505583
2,"LightGBM (n=100, lr=0.1, leaves=31)",1601.689899,1.817195,0.306877
3,Floresta Aleatória (n=80),1636.199291,13.515549,0.398744
4,Floresta Aleatória (n=40),1642.664162,6.886994,0.196559
5,Árvore de Decisão (depth=15),1902.477072,0.507590,0.012220
6,Árvore de Decisão (depth=10),1963.193836,0.369853,0.007864
7,Árvore de Decisão (depth=5),2418.380066,0.303195,0.005378
8,Regressão Linear,2953.269890,0.037547,0.004660


## Conclusões

Comparando os modelos pelos três critérios que a Rusty Bargain quer (qualidade, velocidade de predição e tempo de treino):

**Qualidade (REQM):** o LightGBM teve o menor erro, seguido de perto pela floresta aleatória. A árvore de decisão ficou no meio e a regressão linear foi a pior. O fato da regressão linear ser a pior é justamente a prova real de que os outros modelos estão funcionando: nenhum gradient boosting ficou abaixo dela, então nada deu errado.

**Velocidade de treino:** a regressão linear é a mais rápida de treinar, mas tem qualidade ruim. Entre os bons modelos, o LightGBM treina muito mais rápido que a floresta aleatória, mesmo com mais estimadores, porque é otimizado para isso. A floresta aleatória é a mais lenta de treinar.

**Velocidade de predição:** todos preveem rápido, mas a árvore de decisão e a regressão linear são as mais rápidas. O LightGBM e a floresta aleatória demoram um pouco mais na predição por terem vários estimadores.

**Escolha final:** o LightGBM é o melhor para a Rusty Bargain. Ele tem a melhor qualidade de previsão e treina rápido, o que importa porque o modelo precisará ser retreinado conforme novos carros entram na base. A floresta aleatória chega perto na qualidade, mas demora bem mais para treinar. A regressão linear só serviu de referência.

# Checklist

Digite 'x' para verificar. Em seguida, pressione Shift + Enter.

- [x]  O Jupyter Notebook está aberto
- [x]  O código está livre de erros
- [x]  As células com o código foram organizadas em ordem de execução
- [x]  Os dados foram baixados e preparados
- [x]  Os modelos foram treinados
- [x]  A análise de velocidade e qualidade dos modelos foi realizada